In [ ]:
import gondola as gon
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time


import enum
import importlib.util
import sys
from pathlib import Path


import polars as pl
import numpy as np
import scipy.integrate as integrate
import HErmes as he
import HErmes.fitting as fit
import scipy.stats as st
import matplotlib

from scipy.spatial.transform import Rotation as rot
from datetime import datetime, UTC, timezone
from glob import glob

#pybindings
from pathlib import Path
import dashi as d
d.visual()
import tqdm


import matplotlib.pyplot as plt


run_id = 251
paddles = gon.db.TofPaddle.all()
#data_path = "/mnt/ucla-gaps-nas1/tof-data/antarctica//data"
#dataset = Path(f"{data_path}/{run_id}")
#files = list(dataset.glob("*.tof.gaps"))
files = [
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_10.251225_214949UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_100.251225_222655UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_102.251225_222745UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_110.251225_223103UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_111.251225_223129UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_112.251225_223154UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_114.251225_223243UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_115.251225_223308UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/compressed/10178/Run10178_117.251225_223357UTC.tof.gaps"
]
files = [
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_1345.251207_014209UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_1848.251207_075922UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_124.251206_101749UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_1789.251207_071305UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_860.251206_191532UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_670.251206_165431UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_86.251206_095058UTC.tof.gaps",
    "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/251/Run251_1323.251207_012610UTC.tof.gaps"
]


calib = gon.calibration.load_rb_calibrations(Path("/mnt/ucla-gaps-nas1/tof-data/antarctica/flight_ssd_data/calib/251222_010511UTC"))
calib = gon.calibration.load_rb_calibrations(Path("/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/calib/251123_215129UTC"))

print("nfiles =", len(files))

print("n calib entries =", len(calib))
import numpy as np
import matplotlib.pyplot as plt
import time

baseline_a_rms_vals = []
baseline_b_rms_vals = []

# ---------------------------------
# loop over files / events
# ---------------------------------
endF = 20

for f in files[:endF]:
    print("opening:", f)

    reader = gon.io.TofPacketReader(str(f))
    packcntEv = 0

    for pack in reader:
        if pack.packet_type != gon.packets.TofPacketType.TofEvent:
            continue

        ev = gon.events.TofEvent.from_bytestream(pack.payload, 0)
        packcntEv += 1
        
        for rb in ev.rb_events:
            for hit in rb.hits:
                if hit.paddle_id == 1: 
                    baseline_a_rms_vals.append(hit.baseline_a_rms)
                    baseline_b_rms_vals.append(hit.baseline_b_rms)

                # optional slow-down for debugging
                # time.sleep(1)

print("N baseline A RMS =", len(baseline_a_rms_vals))
print("N baseline B RMS =", len(baseline_b_rms_vals))

baseline_a_rms_vals = np.asarray(baseline_a_rms_vals)
baseline_b_rms_vals = np.asarray(baseline_b_rms_vals)

bins = np.linspace(-1, 4, 501)  # 500 bins from -1 to 4 mV

plt.figure(figsize=(8, 5))
plt.hist(baseline_a_rms_vals, bins=bins, alpha=0.6, label="Side A baseline RMS")
plt.hist(baseline_b_rms_vals, bins=bins, alpha=0.6, label="Side B baseline RMS")
plt.xlabel("Baseline RMS [mV]")
plt.ylabel("Counts")
plt.title("Baseline RMS Distribution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

baseline_a_rms_vals = np.asarray(baseline_a_rms_vals)
baseline_b_rms_vals = np.asarray(baseline_b_rms_vals)

# ---------------------------------
# histogram settings
# ---------------------------------
bins = np.linspace(-1, 4, 500)

# choose which one to display
vals = baseline_a_rms_vals
label = "baseline A RMS"

# ---------------------------------
# stats
# ---------------------------------
mean = np.mean(vals)
std = np.std(vals)
entries = len(vals)

# ---------------------------------
# plot
# ---------------------------------
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    vals,
    bins=bins,
    histtype="step",
    linewidth=1.2
)

ax.set_xlim(-1, 4)

ax.set_xlabel(r"Pedestal RMS Rust (mV)")
ax.set_ylabel("Counts")

ax.set_title(
    r"Ped Sigma Rust paddle 1: "
    r"$\sqrt{\mathrm{sum2}/(N-\mathrm{mean}^2)}$"
)

# ---------------------------------
# ROOT-like stats box
# ---------------------------------
stats_text = (
    f"{label}\n"
    f"Entries      {entries}\n"
    f"Mean         {mean:.4f}\n"
    f"Std Dev      {std:.4f}"
)

ax.text(
    0.985,
    0.98,
    stats_text,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(facecolor="white", edgecolor="black")
)

plt.tight_layout()
plt.show()

In [ ]:

#%run prelude.rc

import enum
import importlib.util
import sys
from pathlib import Path


import polars as pl
import numpy as np
import scipy.integrate as integrate
import HErmes as he
import HErmes.fitting as fit
import scipy.stats as st
import matplotlib

from scipy.spatial.transform import Rotation as rot
from datetime import datetime, UTC, timezone
from glob import glob

#pybindings
from pathlib import Path
import dashi as d
d.visual()
import tqdm


import matplotlib.pyplot as plt
import charmingbeauty as cb
lo = cb.layout
cb.visual.set_style_present()


import re
!export DJANGO_ALLOW_ASYNC_UNSAFE=1
import os
from matplotlib import font_manager
from matplotlib import rcParams


os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'
plt.rcParams.update({'text.usetex' : False})


from matplotlib import font_manager


rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Open Sans']


import gondola as gon
import time

files = gon.io.grace_get_telemetry_binaries(
    1765835400,
    1767979800, #end time
    '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
)


lpt = None
toml_find = False


for f in files:  # in files from flight
    reader = gon.io.TelemetryPacketReader(f)

    for pack in reader:

        if pack.packet_type == gon.packets.TelemetryPacketType.AnyTofHK:
            tp = gon.packets.TofPacket.from_bytestream(pack.payload, 0)

            if tp.packet_type == gon.packets.TofPacketType.LiftofSettings:
                lpt = tp
                toml_find = True

        if toml_find is True:
            if (
                pack.packet_type == gon.packets.TelemetryPacketType.BoringEvent
                or pack.packet_type == gon.packets.TelemetryPacketType.InterestingEvent
            ):

                toml_find = False
                ev = gon.events.TelemetryEvent.from_telemetrypacket(pack)
                
                print("++++ new run ++++")
                print(f)
                print(ev.tof.run_id)
                print(ev.tof.__dir__())
                print(ev.tof.timestamp)
                # Decompress TOML
                gon.io.decompress_toml(lpt.payload, 'test.toml')
                # Open and print first 130 lines
                print("----- toml  -----")
                with open("test2.toml", "r") as toml_file:
                    for i, line in enumerate(toml_file):
                        if i >= 130:
                            break
                        print(line.rstrip())
                print("---------------------------------------")

                # Now process event
                

